# **MODELO DE REGRESIÓN**
# **Bosque Aleatorio (** Random Forest **)**

Este es el **siguiente paso** después del Árbol de Decisión. Si un árbol es bueno, ¿por qué no tener un "bosque" de 100 árboles diferentes y que "voten" (en este caso, promedien) sus predicciones? Esto lo hace mucho más robusto y menos propenso a cometer los errores de un solo árbol.

En lugar de construir un solo árbol (que puede ser propenso a sobreajustarse o a cometer errores específicos), un Random Forest construye cientos de árboles (**n_estimators=100**).

## **Paso 1: Introducción y Librerías**

**Objetivo:** Predecir el valor mediano de una vivienda (median_house_value) en California usando un modelo de Regresión Lineal.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
# Librerías necesarias para el modelo
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.ensemble import RandomForestRegressor

In [ ]:
# Métricas de regresión
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## **Paso 2: Cargar y Explorar los Datos**

Cargaremos el dataset *california_housing_train.csv*.

In [ ]:
# Cargar el dataset
datos = pd.read_csv('https://github.com/estebangonzalezITM/DataScience/raw/main/datasets/california_housing_train.csv')
datos.head()

In [ ]:
# Ver la información de las columnas (tipos de datos, nulos)
datos.info()

In [ ]:
# Mostrar las columnas con datos faltantes
datos.isnull().sum()

## **Paso 3: Preparación de Datos**

El método ***.info()*** mostró que este dataset es "limpio" en el sentido de que no tiene valores nulos (NaN). Cada columna tiene 17,000 entradas completas.

Eso nos ahorra un paso de "limpieza de corrección" (como tener que rellenar o borrar filas).

## **Paso 4: Definir Features (X) y Target (y)**

Ahora usaremos todas las columnas del dataset como caracterìsticas (X) excepto el valor media de la vivienda ('median_house_value'), ya que es nuestra variable objetivo ('y')

In [ ]:
# Definir X e y
X = datos[datos.columns.drop('median_house_value')] # X debe ser un DataFrame (con doble corchete)
y = datos['median_house_value']     # y es una Serie (con un corchete)

## **Paso 5: Dividir los Datos (Train / Test)**

Igual que en clasificación, se deben dividir los datos para poder evaluar el modelo de forma justa.

In [ ]:
# Usaremos los X e y que definimos arriba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tamaño de X_train: {X_train.shape}")
print(f"Tamaño de X_test: {X_test.shape}")

## **Paso 6: Crear y Entrenar el Modelo Random Forest**

El entrenamiento será notablemente más lento que el de un solo árbol, ya que está entrenando 100 de ellos.

*   **n_estimators=100:** Le decimos que construya 100 árboles.
*   **max_depth=5:** Profundidad del arbol de desición.
*   **n_jobs=-1:** (Opcional pero recomendado) ¡Usa todos los núcleos del procesador para entrenar en paralelo y hacerlo más rápido!

In [ ]:
modelo_rf = RandomForestRegressor(n_estimators=100,max_depth=5, random_state=42, n_jobs=-1)

# Entrenar el modelo
modelo_rf.fit(X_train, y_train)

## **Paso 7: Evaluar el Modelo Árbol de Desición**

## **Paso 7.1: Calculamos la métricas RMSE y R²**

Es hora de usar las nuevas métricas, para ello se deben hacer predicciones sobre **X_test** y las compararemos con **y_test**.

* **RMSE (Raíz del Error Cuadrático Medio)**: Similar al MAE, pero penaliza más los errores grandes.

* **R² (R-Cuadrado)**: El porcentaje de la varianza del precio que nuestro modelo pudo "explicar". (0 a 1, más alto es mejor).

In [ ]:
# Hacer predicciones sobre el set de TEST
y_pred = modelo_rf.predict(X_test)

print("--- Métricas del Modelo Lineal Simple ---")

# Calcular las métricas
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print(f"Raíz del Error Cuadrático (RMSE): ${rmse:,.0f}")

r2 = r2_score(y_test, y_pred)
print(f"Coeficiente R-cuadrado (R²):      {r2:.3f}")

***
## **Paso 7.2: Optimizar los hiperparametros (`GridSearchCV`)**

GridSearchCV es la herramienta que permite que en lugar de probar "a mano" qué número de árboles o qué profundidad funciona mejor, dejas que Python lo haga por ti de forma sistemática.

>### **7.2.1. Definimos el diccionario de parámetros a probar**

In [ ]:
parametros = {
    'n_estimators': [100, 250, 500],        # Número de árboles en el bosque
    'max_depth': [5, 10, None],          # Profundidad máxima de los árboles
}

> ### **7.2.2. Inicializamos el modelo base (sin configurar hiperparámetros)**

In [ ]:
rf = RandomForestRegressor(random_state=42)

> ### **7.2.3. Configuramos GridSearchCV**

* `cv=5` significa que usará Cross-Validation de 5 pliegues, es decir, entrena y prueba cada combinación de parámetros 5 veces.

* `n_jobs=-1` usa todos los núcleos de tu procesador para ir más rápido
* `verbose=1` muestra un resumen al inicio y al final

In [ ]:
grid_search = GridSearchCV(estimator=rf, param_grid=parametros, cv=3, n_jobs=-1, verbose=1)

> ### **7.2.4. Entrenamos buscando la mejor combinación**

In [ ]:
grid_search.fit(X_train, y_train)

> ### **7.2.5. Mostramos los resultados del modelo optimizado**

In [ ]:
print("--- Mejores Hiperparámetros encontrados ---")
print(grid_search.best_params_)

print(f"\nMejor puntuación coeficiente R²: {grid_search.best_score_:.3f}")

mejor_modelo = grid_search.best_estimator_

## **Paso 8: Interpretando el Árbol (Importancia de Features)**
**(Opcional)**

Los árboles no tienen pendiente e intercepto como la Regresión Lineal. En su lugar, tienen "Importancia de Features" (**feature_importances_**).

Esto nos dice qué features usó el árbol más a menudo en sus "preguntas" para dividir los datos. Es una forma diferente de medir qué variables fueron las más importantes para la predicción

In [ ]:
# Obtener y ordenar las importancias
importancias_arbol = mejor_modelo.feature_importances_
importancia_arbol_series = pd.Series(importancias_arbol, index=X.columns)
importancia_arbol_ordenada = importancia_arbol_series.sort_values(ascending=False)

print("--- Importancia de las Features (Árbol de Decisión) ---")
print(importancia_arbol_ordenada)

# Graficarlo
plt.figure
sns.barplot(x=importancia_arbol_ordenada.values, y=importancia_arbol_ordenada.index)
plt.title('Importancia de las Features (Árbol de Decisión)')
plt.xlabel('Nivel de Importancia')
plt.ylabel('Feature')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()